Title: LEE_Detection_ERA5_1960_.ipynb

Purpose: Identifiy Low Energy Events with varying lengths from the model output data for all years of ERA5 data

Author: Onno Nennecke on 21.03.2025 Modified: 07.04.2026

Input data: 

- adjusted final model output
    - This file lies here: '/climca/people/onennecke/model_output/not_bias_corrected/full_year/ERA5_all_years/ERA5_hist_timeseries.nc'

Output data:

- LEE Tables: LEE_dat_14.csv, LEE_dat_7.csv, LEE_dat.csv, LEE_dat_14_selection.csv, LEE_dat_7_selection.csv, LEE_vl.csv
    - This file lies here: '/climca/people/onennecke/model_output/LEE_detection/'

In [1]:
# Importing libraries
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.ndimage as ndimage
import os
import glob


### Read Model output data

In [2]:
# path = '/climca/people/onennecke/model_output/not_bias_corrected/model_output_adj.nc'
path = '/climca/people/onennecke/model_output/not_bias_corrected/model_output_all.nc'
# path = '/climca/people/onennecke/model_output/not_bias_corrected/model_output_all_future.nc'
path = '/climca/people/onennecke/model_output/not_bias_corrected/full_year/ERA5_all_years/ERA5_hist_timeseries.nc'

ts_datasets = xr.open_dataset(path)
ts_datasets.load()

<xarray.Dataset> Size: 2MB
Dimensions:        (time: 23725)
Coordinates:
  * time           (time) datetime64[ns] 190kB 1960-01-01 ... 2024-12-31
    crs            int64 8B 4326
    gridtype       <U6 24B 'lonlat'
    ESM            <U9 36B 'ERA5_week'
    run            <U4 16B 'hist'
    ESM_run        <U14 56B 'ERA5_hist_week'
    country        float64 8B 9.0
    period         <U4 16B 'week'
Data variables:
    temp           (time) float64 190kB 7.185 7.429 6.072 ... 0.7396 0.4142
    demand         (time) float64 190kB 1.43e+03 1.427e+03 ... 1.515e+03
    sfcWind        (time) float32 95kB 8.02 5.214 3.839 ... 6.159 8.049 8.775
    rsds           (time) float32 95kB 24.19 16.84 14.44 ... 28.84 26.3 31.92
    tas            (time) float32 95kB 6.337 6.546 5.314 ... 0.2238 1.512 1.194
    tasmax         (time) float32 95kB 7.993 7.899 6.449 ... 2.446 2.709 3.361
    wind_off_prod  (time) float64 190kB 130.4 10.01 6.018 ... 140.7 221.1 221.1
    wind_on_prod   (time) float64 190kB 612.3 163.3 63.56 ... 349.6 783.9 831.2
    solar_prod     (time) float64 190kB 41.56 31.6 21.69 ... 53.04 48.64 59.86
    total_prod     (time) float64 190kB 784.3 204.9 ... 1.054e+03 1.112e+03
    Netto          (time) float64 190kB -645.7 -1.222e+03 ... -457.1 -402.6
    Residual_load  (time) float64 190kB 645.7 1.222e+03 ... 457.1 402.6

In [3]:
# path = '/climca/people/onennecke/model_output/bias_corrected_masked_ibicus/full_year/'
# files = sorted(glob.glob(path + '*.nc'))
# # files = files[:60] + files[61:]
# ts_datasets = xr.open_mfdataset(files, combine='nested', concat_dim = 'ESM_run')
# ts_datasets.coords['doy'] = (('time',), np.tile(np.arange(1, 366), 10))

# ts_datasets

#### Identification of low energy events

In [3]:
# RL = ts_datasets['Residual_load_adj']
RL = ts_datasets['Residual_load']

RL.values

array([ 645.73438129, 1221.94157916, 1353.00669838, ...,  984.9126268 ,
        457.11872085,  402.62437437])

In [4]:
# Calculate the rolling means
ts_datasets['RL_mov_avg_7'] = RL_mov_avg_7 = RL.rolling(time=7, center=False).mean()
ts_datasets['RL_mov_avg_14'] = RL_mov_avg_14 = RL.rolling(time=14, center=False).mean()
# ts_datasets

In [5]:
ts_datasets.load()

<xarray.Dataset> Size: 2MB
Dimensions:        (time: 23725)
Coordinates:
  * time           (time) datetime64[ns] 190kB 1960-01-01 ... 2024-12-31
    crs            int64 8B 4326
    gridtype       <U6 24B 'lonlat'
    ESM            <U9 36B 'ERA5_week'
    run            <U4 16B 'hist'
    ESM_run        <U14 56B 'ERA5_hist_week'
    country        float64 8B 9.0
    period         <U4 16B 'week'
Data variables: (12/14)
    temp           (time) float64 190kB 7.185 7.429 6.072 ... 0.7396 0.4142
    demand         (time) float64 190kB 1.43e+03 1.427e+03 ... 1.515e+03
    sfcWind        (time) float32 95kB 8.02 5.214 3.839 ... 6.159 8.049 8.775
    rsds           (time) float32 95kB 24.19 16.84 14.44 ... 28.84 26.3 31.92
    tas            (time) float32 95kB 6.337 6.546 5.314 ... 0.2238 1.512 1.194
    tasmax         (time) float32 95kB 7.993 7.899 6.449 ... 2.446 2.709 3.361
    ...             ...
    solar_prod     (time) float64 190kB 41.56 31.6 21.69 ... 53.04 48.64 59.86
    total_prod     (time) float64 190kB 784.3 204.9 ... 1.054e+03 1.112e+03
    Netto          (time) float64 190kB -645.7 -1.222e+03 ... -457.1 -402.6
    Residual_load  (time) float64 190kB 645.7 1.222e+03 ... 457.1 402.6
    RL_mov_avg_7   (time) float64 190kB nan nan nan ... 1.171e+03 1.043e+03
    RL_mov_avg_14  (time) float64 190kB nan nan nan nan ... 764.8 806.4 789.2

In [6]:
print(ts_datasets.Residual_load.values[0:10])
print(ts_datasets.RL_mov_avg_7.values[0:10])

[ 645.73438129 1221.94157916 1353.00669838 1322.60320753  984.11530212
  316.36161274  772.2102402  1129.57886782  783.02608938 1383.60529691]
[          nan           nan           nan           nan           nan
           nan  945.13900306 1014.25964399  951.55743117  955.92865953]


##### Threshold Calculation

In [7]:
thresh_perc = 0.95
threshold_week = np.float64(1319.138650687668)
threshold_week

np.float64(1319.138650687668)

In [8]:
# Time series of "True" when threshold is exceeded, "False" otherwise

exceed_bool_1 = xr.where(RL > threshold_week, True, False)
exceed_bool_7 = xr.where(RL_mov_avg_7 > threshold_week, True, False)
exceed_bool_14 = xr.where(RL_mov_avg_14 > threshold_week, True, False)

In [9]:
# Look for events without any rolling mean


# Count number of true values overall
count_exceed_1 = exceed_bool_1.sum(dim='time')
count_exceed_1.values
# exceed_bool_1.time
# np.zeros_like(exceed_bool_1, dtype=int)

array(1368)

### Days above threshold (dat) 

- Take each day as its own event (not really events but just days above threshold)

In [10]:
# 1) Extract the flat indices where mask is True
time_idx = np.nonzero(exceed_bool_1.values)
run_idx = np.repeat(0, len(time_idx[0]))
n_dat = run_idx.size

# 2) Create a flat counter 1…n_dat
labels = np.arange(1, n_dat + 1, dtype=int)

# 3) Scatter them back into an integer array of same shape
dat = np.zeros_like(exceed_bool_1.values, dtype=int)
dat[time_idx] = labels

# wrap back into an xarray
dat = xr.DataArray(
    dat,
    coords=exceed_bool_1.coords,
    dims=exceed_bool_1.dims,
    name="dat"
)

n_dat = n_dat
dat

<xarray.DataArray 'dat' (time: 23725)> Size: 190kB
array([0, 0, 1, ..., 0, 0, 0])
Coordinates:
  * time      (time) datetime64[ns] 190kB 1960-01-01 1960-01-02 ... 2024-12-31
    crs       int64 8B 4326
    gridtype  <U6 24B 'lonlat'
    ESM       <U9 36B 'ERA5_week'
    run       <U4 16B 'hist'
    ESM_run   <U14 56B 'ERA5_hist_week'
    country   float64 8B 9.0
    period    <U4 16B 'week'

In [11]:
# 1) Extract the flat indices where mask is True
time_idx = np.nonzero(exceed_bool_7.values)
run_idx = np.repeat(0, len(time_idx[0]))

n_dat_7 = run_idx.size

# 2) Create a flat counter 1…n_dat_7
labels = np.arange(1, n_dat_7 + 1, dtype=int)

# 3) Scatter them back into an integer array of same shape
dat_7 = np.zeros_like(exceed_bool_7.values, dtype=int)
dat_7[time_idx] = labels

# wrap back into an xarray
dat_7 = xr.DataArray(
    dat_7,
    coords=exceed_bool_7.coords,
    dims=exceed_bool_7.dims,
    name="dat_7"
)

n_dat_7 = n_dat_7
dat_7

<xarray.DataArray 'dat_7' (time: 23725)> Size: 190kB
array([0, 0, 0, ..., 0, 0, 0])
Coordinates:
  * time      (time) datetime64[ns] 190kB 1960-01-01 1960-01-02 ... 2024-12-31
    crs       int64 8B 4326
    gridtype  <U6 24B 'lonlat'
    ESM       <U9 36B 'ERA5_week'
    run       <U4 16B 'hist'
    ESM_run   <U14 56B 'ERA5_hist_week'
    country   float64 8B 9.0
    period    <U4 16B 'week'

In [12]:
# 1) Extract the flat indices where mask is True
time_idx = np.nonzero(exceed_bool_14.values)
run_idx = np.repeat(0, len(time_idx[0]))

n_dat_14 = run_idx.size

# 2) Create a flat counter 1…n_dat_14
labels = np.arange(1, n_dat_14 + 1, dtype=int)

# 3) Scatter them back into an integer array of same shape
dat_14 = np.zeros_like(exceed_bool_14.values, dtype=int)
dat_14[time_idx] = labels

# wrap back into an xarray
dat_14 = xr.DataArray(
    dat_14,
    coords=exceed_bool_14.coords,
    dims=exceed_bool_14.dims,
    name="dat_14"
)

n_dat_14 = n_dat_14
dat_14

<xarray.DataArray 'dat_14' (time: 23725)> Size: 190kB
array([0, 0, 0, ..., 0, 0, 0])
Coordinates:
  * time      (time) datetime64[ns] 190kB 1960-01-01 1960-01-02 ... 2024-12-31
    crs       int64 8B 4326
    gridtype  <U6 24B 'lonlat'
    ESM       <U9 36B 'ERA5_week'
    run       <U4 16B 'hist'
    ESM_run   <U14 56B 'ERA5_hist_week'
    country   float64 8B 9.0
    period    <U4 16B 'week'

### Events with rolling mean of 1 above threshold (events_vl) = events with varying length 

In [13]:
# Look for events without any rolling mean

events_vl = np.zeros_like(exceed_bool_1, dtype=int)


labeled_segment, num_features = ndimage.label(exceed_bool_1.values)
current_label = 1

if num_features > 0:
    labeled_segment[labeled_segment > 0] += current_label - 1
    current_label += num_features
events_vl = labeled_segment

# # Starting value for the labels
# current_label = 1
# counter = 0
# for run in exceed_bool_1.ESM_run.values:
#     run_data = exceed_bool_1.sel(ESM_run=run)
#     # Only label this section
#     labeled_segment, num_features = ndimage.label(run_data.values)

#     if num_features > 0:
#         labeled_segment[labeled_segment > 0] += current_label - 1
#         current_label += num_features

#     # Save the label to the result array
#     events_vl[counter] = labeled_segment
#     counter += 1

n_events_vl = current_label - 1
n_events_vl



756

In [14]:
# def LEE_detection(events, n_events, t=RL['time'].values, dur = 7, minDuration=1):
    
# events = dat
# n_events = n_dat
# t=RL['time'].values
# dur = 1
# minDuration=1




def LEE_detection(events, n_events, t=RL['time'].values, dur = 7, minDuration=1):
    LEE_records = []
    for ev in range(1, n_events + 1):
        event_duration = (events == ev).sum()
        if event_duration < minDuration:
            continue

        end_idx = np.where(events == ev)[0][0]
        start_idx = end_idx - dur + 1

        date_start = t[start_idx]
        date_end = t[end_idx]

        LEE_start = np.where(t == date_start)[0][0]
        LEE_end = np.where(t == date_end)[0][0]

        RL_run = RL.values
        RL_LEE = RL_run[LEE_start:LEE_end + 1]
        LEE_peak = np.argmax(RL_LEE)

        record = {
            'date_start': date_start,
            'date_end': date_end,
            'date_peak': date_start + LEE_peak,
            # 'date_start_old': RL['old_time'][i][LEE_start].values,
            # 'date_end_old': RL['old_time'][i][LEE_end].values,
            # 'date_peak_old': RL['old_time'][i][LEE_start + LEE_peak].values,
            'index_start': LEE_start,
            'index_end': LEE_end,
            'index_peak': LEE_start + LEE_peak,
            'duration': len(RL_LEE),
            'RL_max': RL_LEE[LEE_peak],
            'RL_mean': RL_LEE.mean(),
            'RL_var': np.sqrt(RL_LEE.var()),
            'RL_cumulative': RL_LEE.sum(),
            'event': ev,
            'ESM': str(RL.ESM.values),
            'ESM_run': str(RL.ESM_run.values),
            'year': RL.time.dt.year[start_idx].values,
            # 'doy' : RL.doy[start_idx].values#,
            # 'winter': RL.winter_year[start_idx].values,
            # 'day_of_winter': RL.day_of_winter[start_idx].values
        }

        # # Optional additional metrics if available
        # # (you could generalize this to loop over var names too)
        # for var in ['prod', 'demand', 'pot']:
        #     try:
        #         var_data = RL[var].sel(ESM_run=RL.ESM_run[i]).values[LEE_start:LEE_end + 1]
        #         record[f'{var}_max'] = var_data.max()
        #         record[f'{var}_mean'] = var_data.mean()
        #         record[f'{var}_var'] = np.sqrt(var_data.var())
        #         record[f'{var}_cumulative'] = var_data.sum()
        #     except KeyError:
        #         pass  # Variable doesn't exist, skip

        LEE_records.append(record)

    LEE_dat = pd.DataFrame(LEE_records)
    return LEE_dat


In [15]:
LEE_dat = LEE_detection(dat, n_dat, dur = 1)

In [16]:
LEE_dat

,date_start,date_end,date_peak,index_start,index_end,index_peak,duration,RL_max,RL_mean,RL_var,RL_cumulative,event,ESM,ESM_run,year
0,1960-01-03,1960-01-03,1960-01-03,2,2,2,1,1353.006698,1353.006698,0.0,1353.006698,1,ERA5_week,ERA5_hist_week,1960
1,1960-01-04,1960-01-04,1960-01-04,3,3,3,1,1322.603208,1322.603208,0.0,1322.603208,2,ERA5_week,ERA5_hist_week,1960
2,1960-01-10,1960-01-10,1960-01-10,9,9,9,1,1383.605297,1383.605297,0.0,1383.605297,3,ERA5_week,ERA5_hist_week,1960
3,1960-01-12,1960-01-12,1960-01-12,11,11,11,1,1356.199583,1356.199583,0.0,1356.199583,4,ERA5_week,ERA5_hist_week,1960
4,1960-01-15,1960-01-15,1960-01-15,14,14,14,1,1470.964648,1470.964648,0.0,1470.964648,5,ERA5_week,ERA5_hist_week,1960
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1363,2024-12-12,2024-12-12,2024-12-12,23705,23705,23705,1,1441.693483,1441.693483,0.0,1441.693483,1364,ERA5_week,ERA5_hist_week,2024
1364,2024-12-13,2024-12-13,2024-12-13,23706,23706,23706,1,1381.942848,1381.942848,0.0,1381.942848,1365,ERA5_week,ERA5_hist_week,2024
1365,2024-12-26,2024-12-26,2024-12-26,23719,23719,23719,1,1371.499075,1371.499075,0.0,1371.499075,1366,ERA5_week,ERA5_hist_week,2024
1366,2024-12-27,2024-12-27,2024-12-27,23720,23720,23720,1,1424.498608,1424.498608,0.0,1424.498608,1367,ERA5_week,ERA5_hist_week,2024


In [17]:
LEE_dat.to_csv('/climca/people/onennecke/model_output/LEE_detection/LEE_dat_ERA5_1960_.csv', index=False)
# LEE_dat.to_csv('/climca/people/onennecke/model_output/LEE_detection/not_bc_adj/LEE_dat.csv', index=False)

LEE_dat

,date_start,date_end,date_peak,index_start,index_end,index_peak,duration,RL_max,RL_mean,RL_var,RL_cumulative,event,ESM,ESM_run,year
0,1960-01-03,1960-01-03,1960-01-03,2,2,2,1,1353.006698,1353.006698,0.0,1353.006698,1,ERA5_week,ERA5_hist_week,1960
1,1960-01-04,1960-01-04,1960-01-04,3,3,3,1,1322.603208,1322.603208,0.0,1322.603208,2,ERA5_week,ERA5_hist_week,1960
2,1960-01-10,1960-01-10,1960-01-10,9,9,9,1,1383.605297,1383.605297,0.0,1383.605297,3,ERA5_week,ERA5_hist_week,1960
3,1960-01-12,1960-01-12,1960-01-12,11,11,11,1,1356.199583,1356.199583,0.0,1356.199583,4,ERA5_week,ERA5_hist_week,1960
4,1960-01-15,1960-01-15,1960-01-15,14,14,14,1,1470.964648,1470.964648,0.0,1470.964648,5,ERA5_week,ERA5_hist_week,1960
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1363,2024-12-12,2024-12-12,2024-12-12,23705,23705,23705,1,1441.693483,1441.693483,0.0,1441.693483,1364,ERA5_week,ERA5_hist_week,2024
1364,2024-12-13,2024-12-13,2024-12-13,23706,23706,23706,1,1381.942848,1381.942848,0.0,1381.942848,1365,ERA5_week,ERA5_hist_week,2024
1365,2024-12-26,2024-12-26,2024-12-26,23719,23719,23719,1,1371.499075,1371.499075,0.0,1371.499075,1366,ERA5_week,ERA5_hist_week,2024
1366,2024-12-27,2024-12-27,2024-12-27,23720,23720,23720,1,1424.498608,1424.498608,0.0,1424.498608,1367,ERA5_week,ERA5_hist_week,2024


In [18]:
LEE_dat_7 = LEE_detection(dat_7, n_dat_7, dur = 7)

In [19]:
# If the date_start is the the next day from the line before and they are the same ESM_run they should get the same event number
new_run = LEE_dat_7['ESM_run'].ne(LEE_dat_7['ESM_run'].shift())
not_consecutive = (LEE_dat_7['date_start'] - LEE_dat_7['date_start'].shift()) != pd.Timedelta(days=1)

# Combine them — whenever either is True, that row is the start of a new event
is_new_event = new_run | not_consecutive

# 4. Cum-sum that to get a 1-based event ID
LEE_dat_7['event'] = is_new_event.cumsum()
LEE_dat_7

,date_start,date_end,date_peak,index_start,index_end,index_peak,duration,RL_max,RL_mean,RL_var,RL_cumulative,event,ESM,ESM_run,year
0,1960-12-12,1960-12-18,1960-12-12 00:00:00.000000001,345,351,346,7,1456.974852,1355.809001,88.578049,9490.663009,1,ERA5_week,ERA5_hist_week,1960
1,1960-12-13,1960-12-19,1960-12-13 00:00:00.000000000,346,352,346,7,1456.974852,1346.571485,92.437264,9426.000398,1,ERA5_week,ERA5_hist_week,1960
2,1960-12-14,1960-12-20,1960-12-14 00:00:00.000000002,347,353,349,7,1436.721202,1334.812747,82.327679,9343.689230,1,ERA5_week,ERA5_hist_week,1960
3,1960-12-15,1960-12-21,1960-12-15 00:00:00.000000001,348,354,349,7,1436.721202,1369.204004,53.839561,9584.428025,1,ERA5_week,ERA5_hist_week,1960
4,1960-12-16,1960-12-22,1960-12-16 00:00:00.000000000,349,355,349,7,1436.721202,1343.837306,90.992580,9406.861141,1,ERA5_week,ERA5_hist_week,1960
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
275,2022-12-12,2022-12-18,2022-12-12 00:00:00.000000004,22975,22981,22979,7,1486.441387,1334.165906,143.007150,9339.161343,75,ERA5_week,ERA5_hist_week,2022
276,2023-01-22,2023-01-28,2023-01-22 00:00:00.000000004,23016,23022,23020,7,1388.191706,1328.462029,73.743844,9299.234201,76,ERA5_week,ERA5_hist_week,2023
277,2023-11-26,2023-12-02,2023-11-26 00:00:00.000000006,23324,23330,23330,7,1463.499258,1351.249571,105.754763,9458.746999,77,ERA5_week,ERA5_hist_week,2023
278,2023-11-27,2023-12-03,2023-11-27 00:00:00.000000005,23325,23331,23330,7,1463.499258,1356.343244,103.949462,9494.402708,77,ERA5_week,ERA5_hist_week,2023


In [20]:
LEE_dat_7.to_csv('/climca/people/onennecke/model_output/LEE_detection/LEE_dat_7_ERA5_1960_.csv', index=False)
# LEE_dat_7.to_csv('/climca/people/onennecke/model_output/LEE_detection/not_bc_adj/LEE_dat_7.csv', index=False)

LEE_dat_7

,date_start,date_end,date_peak,index_start,index_end,index_peak,duration,RL_max,RL_mean,RL_var,RL_cumulative,event,ESM,ESM_run,year
0,1960-12-12,1960-12-18,1960-12-12 00:00:00.000000001,345,351,346,7,1456.974852,1355.809001,88.578049,9490.663009,1,ERA5_week,ERA5_hist_week,1960
1,1960-12-13,1960-12-19,1960-12-13 00:00:00.000000000,346,352,346,7,1456.974852,1346.571485,92.437264,9426.000398,1,ERA5_week,ERA5_hist_week,1960
2,1960-12-14,1960-12-20,1960-12-14 00:00:00.000000002,347,353,349,7,1436.721202,1334.812747,82.327679,9343.689230,1,ERA5_week,ERA5_hist_week,1960
3,1960-12-15,1960-12-21,1960-12-15 00:00:00.000000001,348,354,349,7,1436.721202,1369.204004,53.839561,9584.428025,1,ERA5_week,ERA5_hist_week,1960
4,1960-12-16,1960-12-22,1960-12-16 00:00:00.000000000,349,355,349,7,1436.721202,1343.837306,90.992580,9406.861141,1,ERA5_week,ERA5_hist_week,1960
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
275,2022-12-12,2022-12-18,2022-12-12 00:00:00.000000004,22975,22981,22979,7,1486.441387,1334.165906,143.007150,9339.161343,75,ERA5_week,ERA5_hist_week,2022
276,2023-01-22,2023-01-28,2023-01-22 00:00:00.000000004,23016,23022,23020,7,1388.191706,1328.462029,73.743844,9299.234201,76,ERA5_week,ERA5_hist_week,2023
277,2023-11-26,2023-12-02,2023-11-26 00:00:00.000000006,23324,23330,23330,7,1463.499258,1351.249571,105.754763,9458.746999,77,ERA5_week,ERA5_hist_week,2023
278,2023-11-27,2023-12-03,2023-11-27 00:00:00.000000005,23325,23331,23330,7,1463.499258,1356.343244,103.949462,9494.402708,77,ERA5_week,ERA5_hist_week,2023


In [21]:
# find the index of the row with the highest RL_cumulative in each event
idx = LEE_dat_7.groupby('event')['RL_cumulative'].idxmax()

# select only those rows
LEE_dat_7_max_cum_RL = LEE_dat_7.loc[idx].reset_index(drop=True)
LEE_dat_7_max_cum_RL


,date_start,date_end,date_peak,index_start,index_end,index_peak,duration,RL_max,RL_mean,RL_var,RL_cumulative,event,ESM,ESM_run,year
0,1960-12-15,1960-12-21,1960-12-15 00:00:00.000000001,348,354,349,7,1436.721202,1369.204004,53.839561,9584.428025,1,ERA5_week,ERA5_hist_week,1960
1,1961-12-16,1961-12-22,1961-12-16 00:00:00.000000002,714,720,716,7,1478.930899,1344.541868,133.358451,9411.793076,2,ERA5_week,ERA5_hist_week,1961
2,1961-12-22,1961-12-28,1961-12-22 00:00:00.000000004,720,726,724,7,1546.163216,1400.927034,85.013603,9806.489237,3,ERA5_week,ERA5_hist_week,1961
3,1962-11-21,1962-11-27,1962-11-21 00:00:00.000000000,1054,1060,1054,7,1442.117243,1327.848696,124.054579,9294.940874,4,ERA5_week,ERA5_hist_week,1962
4,1962-12-01,1962-12-07,1962-12-01 00:00:00.000000004,1064,1070,1068,7,1420.932325,1340.073373,74.116544,9380.513610,5,ERA5_week,ERA5_hist_week,1962
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72,2019-01-19,2019-01-25,2019-01-19 00:00:00.000000005,21553,21559,21558,7,1470.620258,1357.392114,97.549782,9501.744800,73,ERA5_week,ERA5_hist_week,2019
73,2020-11-26,2020-12-02,2020-11-26 00:00:00.000000006,22229,22235,22235,7,1395.032961,1337.845966,54.717352,9364.921762,74,ERA5_week,ERA5_hist_week,2020
74,2022-12-10,2022-12-16,2022-12-10 00:00:00.000000006,22973,22979,22979,7,1486.441387,1413.245665,35.261727,9892.719655,75,ERA5_week,ERA5_hist_week,2022
75,2023-01-22,2023-01-28,2023-01-22 00:00:00.000000004,23016,23022,23020,7,1388.191706,1328.462029,73.743844,9299.234201,76,ERA5_week,ERA5_hist_week,2023


In [22]:
# LEE_dat_7_max_cum_RL.to_csv('/climca/people/onennecke/model_output/LEE_detection/not_bc_adj/LEE_dat_7_selection.csv', index=False)
LEE_dat_7_max_cum_RL.to_csv('/climca/people/onennecke/model_output/LEE_detection/LEE_dat_7_selection_ERA5_1960_.csv', index=False)


In [23]:
LEE_dat_14 = LEE_detection(dat_14, n_dat_14, dur = 14)

In [24]:
LEE_dat_14

,date_start,date_end,date_peak,index_start,index_end,index_peak,duration,RL_max,RL_mean,RL_var,RL_cumulative,event,ESM,ESM_run,year
0,1961-12-15,1961-12-28,1961-12-15 00:00:00.000000011,713,726,724,14,1546.163216,1366.152004,118.579159,19126.128051,1,ERA5_week,ERA5_hist_week,1961
1,1961-12-16,1961-12-29,1961-12-16 00:00:00.000000010,714,727,724,14,1546.163216,1354.012762,135.711545,18956.178667,2,ERA5_week,ERA5_hist_week,1961
2,1963-12-04,1963-12-17,1963-12-04 00:00:00.000000013,1432,1445,1445,14,1501.334083,1347.300440,208.668489,18862.206163,3,ERA5_week,ERA5_hist_week,1963
3,1963-12-05,1963-12-18,1963-12-05 00:00:00.000000012,1433,1446,1445,14,1501.334083,1359.813794,206.650077,19037.393121,4,ERA5_week,ERA5_hist_week,1963
4,1963-12-06,1963-12-19,1963-12-06 00:00:00.000000011,1434,1447,1445,14,1501.334083,1337.158841,228.330735,18720.223773,5,ERA5_week,ERA5_hist_week,1963
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68,2007-12-11,2007-12-24,2007-12-11 00:00:00.000000010,17499,17512,17509,14,1464.791306,1336.984492,109.543102,18717.782882,69,ERA5_week,ERA5_hist_week,2007
69,2017-01-15,2017-01-28,2017-01-15 00:00:00.000000008,20819,20832,20827,14,1470.397466,1319.846111,120.772067,18477.845556,70,ERA5_week,ERA5_hist_week,2017
70,2022-12-03,2022-12-16,2022-12-03 00:00:00.000000013,22966,22979,22979,14,1486.441387,1329.593423,119.606781,18614.307921,71,ERA5_week,ERA5_hist_week,2022
71,2022-12-04,2022-12-17,2022-12-04 00:00:00.000000012,22967,22980,22979,14,1486.441387,1337.666083,112.491283,18727.325164,72,ERA5_week,ERA5_hist_week,2022


In [25]:
# If the date_start is the the next day from the line before and they are the same ESM_run they should get the same event number
new_run = LEE_dat_14['ESM_run'].ne(LEE_dat_14['ESM_run'].shift())
not_consecutive = (LEE_dat_14['date_start'] - LEE_dat_14['date_start'].shift()) != pd.Timedelta(days=1)

# Combine them — whenever either is True, that row is the start of a new event
is_new_event = new_run | not_consecutive

# 4. Cum-sum that to get a 1-based event ID
LEE_dat_14['event'] = is_new_event.cumsum()
# LEE_dat_14

In [26]:
LEE_dat_14.to_csv('/climca/people/onennecke/model_output/LEE_detection/LEE_dat_14_ERA5_1960_.csv', index=False)
# LEE_dat_14.to_csv('/climca/people/onennecke/model_output/LEE_detection/not_bc_adj/LEE_dat_14.csv', index=False)

LEE_dat_14

,date_start,date_end,date_peak,index_start,index_end,index_peak,duration,RL_max,RL_mean,RL_var,RL_cumulative,event,ESM,ESM_run,year
0,1961-12-15,1961-12-28,1961-12-15 00:00:00.000000011,713,726,724,14,1546.163216,1366.152004,118.579159,19126.128051,1,ERA5_week,ERA5_hist_week,1961
1,1961-12-16,1961-12-29,1961-12-16 00:00:00.000000010,714,727,724,14,1546.163216,1354.012762,135.711545,18956.178667,1,ERA5_week,ERA5_hist_week,1961
2,1963-12-04,1963-12-17,1963-12-04 00:00:00.000000013,1432,1445,1445,14,1501.334083,1347.300440,208.668489,18862.206163,2,ERA5_week,ERA5_hist_week,1963
3,1963-12-05,1963-12-18,1963-12-05 00:00:00.000000012,1433,1446,1445,14,1501.334083,1359.813794,206.650077,19037.393121,2,ERA5_week,ERA5_hist_week,1963
4,1963-12-06,1963-12-19,1963-12-06 00:00:00.000000011,1434,1447,1445,14,1501.334083,1337.158841,228.330735,18720.223773,2,ERA5_week,ERA5_hist_week,1963
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68,2007-12-11,2007-12-24,2007-12-11 00:00:00.000000010,17499,17512,17509,14,1464.791306,1336.984492,109.543102,18717.782882,17,ERA5_week,ERA5_hist_week,2007
69,2017-01-15,2017-01-28,2017-01-15 00:00:00.000000008,20819,20832,20827,14,1470.397466,1319.846111,120.772067,18477.845556,18,ERA5_week,ERA5_hist_week,2017
70,2022-12-03,2022-12-16,2022-12-03 00:00:00.000000013,22966,22979,22979,14,1486.441387,1329.593423,119.606781,18614.307921,19,ERA5_week,ERA5_hist_week,2022
71,2022-12-04,2022-12-17,2022-12-04 00:00:00.000000012,22967,22980,22979,14,1486.441387,1337.666083,112.491283,18727.325164,19,ERA5_week,ERA5_hist_week,2022


In [27]:
# find the index of the row with the highest RL_cumulative in each event
idx = LEE_dat_14.groupby('event')['RL_cumulative'].idxmax()

# select only those rows
LEE_dat_14_max_cum_RL = LEE_dat_14.loc[idx].reset_index(drop=True)
LEE_dat_14_max_cum_RL

,date_start,date_end,date_peak,index_start,index_end,index_peak,duration,RL_max,RL_mean,RL_var,RL_cumulative,event,ESM,ESM_run,year
0,1961-12-15,1961-12-28,1961-12-15 00:00:00.000000011,713,726,724,14,1546.163216,1366.152004,118.579159,19126.128051,1,ERA5_week,ERA5_hist_week,1961
1,1963-12-05,1963-12-18,1963-12-05 00:00:00.000000012,1433,1446,1445,14,1501.334083,1359.813794,206.650077,19037.393121,2,ERA5_week,ERA5_hist_week,1963
2,1964-01-07,1964-01-20,1964-01-07 00:00:00.000000005,1466,1479,1471,14,1484.721318,1360.195995,85.192528,19042.743925,3,ERA5_week,ERA5_hist_week,1964
3,1964-12-16,1964-12-29,1964-12-16 00:00:00.000000008,1809,1822,1817,14,1476.479834,1346.187921,103.924323,18846.630888,4,ERA5_week,ERA5_hist_week,1964
4,1969-11-30,1969-12-13,1969-11-30 00:00:00.000000006,3618,3631,3624,14,1503.532015,1377.346133,111.862006,19282.845863,5,ERA5_week,ERA5_hist_week,1969
5,1969-12-04,1969-12-17,1969-12-04 00:00:00.000000002,3622,3635,3624,14,1503.532015,1333.069833,205.717185,18662.977658,6,ERA5_week,ERA5_hist_week,1969
6,1972-12-31,1973-01-13,1972-12-31 00:00:00.000000004,4744,4757,4748,14,1462.266108,1377.143148,62.628939,19280.004074,7,ERA5_week,ERA5_hist_week,1972
7,1980-01-06,1980-01-19,1980-01-06 00:00:00.000000009,7305,7318,7314,14,1461.025588,1368.754154,62.816240,19162.558158,8,ERA5_week,ERA5_hist_week,1980
8,1981-12-15,1981-12-28,1981-12-15 00:00:00.000000002,8013,8026,8015,14,1507.900079,1379.005028,85.096147,19306.070388,9,ERA5_week,ERA5_hist_week,1981
9,1982-01-11,1982-01-24,1982-01-11 00:00:00.000000000,8040,8053,8040,14,1454.308717,1373.493451,68.110456,19228.908315,10,ERA5_week,ERA5_hist_week,1982


In [28]:
# LEE_dat_14_max_cum_RL.to_csv('/climca/people/onennecke/model_output/LEE_detection/not_bc_adj/LEE_dat_14_selection.csv', index=False)
LEE_dat_14_max_cum_RL.to_csv('/climca/people/onennecke/model_output/LEE_detection/LEE_dat_14_selection_ERA5_1960_.csv', index=False)


### Event detection for variable length

In [29]:
def LEE_detection_vl(events, n_events, t=RL['time'].values, minDuration=1):
    LEE_records = []

    for ev in range(1, n_events + 1):
        event_duration = (events == ev).sum()
        if event_duration < minDuration:
            continue

        start_idx = np.where(events == ev)[0][0]
        end_idx = np.where(events == ev)[0][-1]

        date_start = t[start_idx]
        date_end = t[end_idx]

        LEE_start = np.where(t == date_start)[0][0]
        LEE_end = np.where(t == date_end)[0][0]

        RL_run = RL.values
        RL_LEE = RL_run[LEE_start:LEE_end + 1]
        LEE_peak = np.argmax(RL_LEE)

        record = {
            'date_start': date_start,
            'date_end': date_end,
            'date_peak': date_start + LEE_peak,
            # 'date_start_old': RL['old_time'][i][LEE_start].values,
            # 'date_end_old': RL['old_time'][i][LEE_end].values,
            # 'date_peak_old': RL['old_time'][i][LEE_start + LEE_peak].values,
            'index_start': LEE_start,
            'index_end': LEE_end,
            'index_peak': LEE_start + LEE_peak,
            'duration': len(RL_LEE),
            'RL_max': RL_LEE[LEE_peak],
            'RL_mean': RL_LEE.mean(),
            'RL_var': np.sqrt(RL_LEE.var()),
            'RL_cumulative': RL_LEE.sum(),
            'event': ev,
            'ESM': str(RL.ESM.values),
            'ESM_run': str(RL.ESM_run.values),
            'year': RL.time.dt.year[start_idx].values,
            # 'doy' : RL.doy[start_idx].values#,
            # 'winter': RL.winter_year[start_idx].values,
            # 'day_of_winter': RL.day_of_winter[start_idx].values
        }

        # # Optional additional metrics if available
        # # (you could generalize this to loop over var names too)
        # for var in ['prod', 'demand', 'pot']:
        #     try:
        #         var_data = RL[var].sel(ESM_run=RL.ESM_run[i]).values[LEE_start:LEE_end + 1]
        #         record[f'{var}_max'] = var_data.max()
        #         record[f'{var}_mean'] = var_data.mean()
        #         record[f'{var}_var'] = np.sqrt(var_data.var())
        #         record[f'{var}_cumulative'] = var_data.sum()
        #     except KeyError:
        #         pass  # Variable doesn't exist, skip

        LEE_records.append(record)

    return pd.DataFrame(LEE_records)

In [30]:
LEE_vl = LEE_detection_vl(events_vl, n_events_vl)

In [31]:
# LEE_vl.to_csv('/climca/people/onennecke/model_output/LEE_detection/not_bc_adj/LEE_vl.csv', index=False)
LEE_vl.to_csv('/climca/people/onennecke/model_output/LEE_detection/LEE_vl_ERA5_1960_.csv', index=False)


In [32]:
LEE_vl

,date_start,date_end,date_peak,index_start,index_end,index_peak,duration,RL_max,RL_mean,RL_var,RL_cumulative,event,ESM,ESM_run,year
0,1960-01-03,1960-01-04,1960-01-03 00:00:00.000000000,2,3,2,2,1353.006698,1337.804953,15.201745,2675.609906,1,ERA5_week,ERA5_hist_week,1960
1,1960-01-10,1960-01-10,1960-01-10 00:00:00.000000000,9,9,9,1,1383.605297,1383.605297,0.000000,1383.605297,2,ERA5_week,ERA5_hist_week,1960
2,1960-01-12,1960-01-12,1960-01-12 00:00:00.000000000,11,11,11,1,1356.199583,1356.199583,0.000000,1356.199583,3,ERA5_week,ERA5_hist_week,1960
3,1960-01-15,1960-01-17,1960-01-15 00:00:00.000000002,14,16,16,3,1481.843521,1465.722614,15.745241,4397.167841,4,ERA5_week,ERA5_hist_week,1960
4,1960-01-21,1960-01-21,1960-01-21 00:00:00.000000000,20,20,20,1,1395.451152,1395.451152,0.000000,1395.451152,5,ERA5_week,ERA5_hist_week,1960
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
751,2024-11-06,2024-11-07,2024-11-06 00:00:00.000000000,23669,23670,23669,2,1371.963671,1367.486423,4.477248,2734.972846,752,ERA5_week,ERA5_hist_week,2024
752,2024-11-10,2024-11-10,2024-11-10 00:00:00.000000000,23673,23673,23673,1,1345.188399,1345.188399,0.000000,1345.188399,753,ERA5_week,ERA5_hist_week,2024
753,2024-11-29,2024-11-29,2024-11-29 00:00:00.000000000,23692,23692,23692,1,1362.511620,1362.511620,0.000000,1362.511620,754,ERA5_week,ERA5_hist_week,2024
754,2024-12-11,2024-12-13,2024-12-11 00:00:00.000000001,23704,23706,23705,3,1441.693483,1418.225361,26.021671,4254.676082,755,ERA5_week,ERA5_hist_week,2024
